In [1]:
import numpy as np
from numpy import genfromtxt
import scipy.io
from pydot import graph_from_dot_data
from sklearn.tree import export_graphviz
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pandas as pd
import io

from decision_tree_starter import DecisionTree, RandomForest
from decision_tree_starter import preprocess

import random
random.seed(246810)
np.random.seed(246810)

# 1. Spam Dataset

In [2]:
# load spam dataset

dataset = "spam"

if dataset == "spam":
    features = [
        "pain", "private", "bank", "money", "drug", "spam", "prescription",
        "creative", "height", "featured", "differ", "width", "other",
        "energy", "business", "message", "volumes", "revision", "path",
        "meter", "memo", "planning", "pleased", "record", "out",
        "semicolon", "dollar", "sharp", "exclamation", "parenthesis",
        "square_bracket", "ampersand"
    ]
    assert len(features) == 32

    # Load spam data
    path_train = 'datasets/spam_data/spam_data.mat'
    data = scipy.io.loadmat(path_train)
    X = data['training_data']
    y = np.squeeze(data['training_labels'])
    Z = data['test_data']
    class_names = ["Ham", "Spam"]
else:
    raise NotImplementedError("Dataset %s not handled" % dataset)

print("Features", features)
print("Train/test size", X.shape, Z.shape)

Features ['pain', 'private', 'bank', 'money', 'drug', 'spam', 'prescription', 'creative', 'height', 'featured', 'differ', 'width', 'other', 'energy', 'business', 'message', 'volumes', 'revision', 'path', 'meter', 'memo', 'planning', 'pleased', 'record', 'out', 'semicolon', 'dollar', 'sharp', 'exclamation', 'parenthesis', 'square_bracket', 'ampersand']
Train/test size (5629, 32) (5400, 32)


## My Decision Tree

In [ ]:
# Train the best model on the Decision Tree with tuning the "max_depth" hyperparameter

depths_dt = np.arange(1, 41) # Trying max_depth from 1 to 40
train_accs = []
val_accs = []
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
for depth in depths_dt:
    params_dt = {
        "max_depth": depth,
    }
    clf_dt = DecisionTree(**params_dt)
    clf_dt.fit(X_train, y_train)
    y_train_pred = clf_dt.predict(X_train)
    train_accuracy = np.mean(y_train_pred == y_train)
    train_accs.append(train_accuracy)
    y_val_pred = clf_dt.predict(X_val)
    val_accuracy = np.mean(y_val_pred == y_val)
    val_accs.append(val_accuracy)

plt.figure()
plt.plot(depths_dt, train_accs, label='Train Accuracy', c='red')
plt.plot(depths_dt, val_accs, label='Validation Accuracy', c='blue')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Decision Tree: Train/Val Acc vs Max Depth')
plt.legend()
plt.savefig('decision_tree_depth_tuning.png')

In [4]:
# my best decision tree with max_depth = 14

best_depth = 14
params_best_dt = {
    "max_depth": best_depth,
}
clf_dt = DecisionTree(**params_best_dt)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
clf_dt.fit(X_train, y_train)
y_train_pred = clf_dt.predict(X_train)
train_accuracy = np.mean(y_train_pred == y_train)
train_accs.append(train_accuracy)
y_val_pred = clf_dt.predict(X_val)
val_accuracy = np.mean(y_val_pred == y_val)
val_accs.append(val_accuracy)
print("Best decision tree train acc for spam dataset: %.4f" % train_accuracy)
print("Best decision tree val acc for spam dataset: %.4f" % val_accuracy)

Best decision tree train acc for spam dataset: 0.8728
Best decision tree val acc for spam dataset: 0.8188


## My Random Forest

In [5]:
# Train the best model on the Random Forest with hyperparameter tuning

clf_rf = RandomForest()
depths = [5, 7, 9, 11, 13, 15, 17, 20, 25, 30, 35, 40]
param_grid = {
    'params': [{'max_depth': depth} for depth in depths],
    'n': [30, 50, 100, 150, 200],
    'm': [3, 6, 10, 15]
}

grid_search = GridSearchCV(estimator=clf_rf, param_grid=param_grid, scoring='accuracy', cv=5, n_jobs=-1, verbose=1)
grid_search.fit(X, y)

print("Best params:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

Fitting 5 folds for each of 240 candidates, totalling 1200 fits
Best params: {'m': 15, 'n': 150, 'params': {'max_depth': 20}}
Best score: 0.8424211565028615


In [ ]:
# my best decision tree with max_depth = 20, n (num_trees) = 150, m (max_features) = 15
best_m, best_n, params_best_rf = grid_search.best_params_['m'], grid_search.best_params_['n'], grid_search.best_params_['params']

clf_rf = RandomForest(params=params_best_rf, n=best_n, m=best_m)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
clf_rf.fit(X_train, y_train)
y_train_pred = clf_rf.predict(X_train)
train_accuracy = np.mean(y_train_pred == y_train)
y_val_pred = clf_rf.predict(X_val)
val_accuracy = np.mean(y_val_pred == y_val)
print("Best random forest train acc for spam dataset: %.4f" % train_accuracy)
print("Best random forest val acc for spam dataset: %.4f" % val_accuracy)

Best random forest train acc for spam dataset: 0.8932
Best random forest val acc for spam dataset: 0.8250


## Kaggle

In [7]:
# Here I choose Random Forest as my classifier
clf_spam = RandomForest(params=params_best_rf, n=best_n, m=best_m)
clf_spam.fit(X, y)
y_test_pred = clf_spam.predict(Z)

# save the results into a kaggle accepted csv

def results_to_csv(y_test):
    y_test = y_test.astype(int)
    df = pd.DataFrame({'Category': y_test})
    df.index += 1 # Ensures that the index starts at 1
    df.to_csv('submission_spam.csv', index_label='Id')

results_to_csv(y_test_pred)

# 2. Titanic Dataset

In [8]:
# load titanic dataset
dataset = "titanic"

if dataset == "titanic":
    # Load titanic data
    path_train = 'datasets/titanic/titanic_training.csv'
    data = genfromtxt(path_train, delimiter=',', dtype=None)
    path_test = 'datasets/titanic/titanic_testing_data.csv'
    test_data = genfromtxt(path_test, delimiter=',', dtype=None)
    y = data[1:, 0]  # label = survived
    class_names = ["Died", "Survived"]

    labeled_idx = np.where(y != b'')[0]
    y = np.array(y[labeled_idx], dtype=float).astype(int)
    print("Preprocessing the titanic dataset\n")
    X, onehot_features = preprocess(data[1:, 1:], onehot_cols=[1, 5, 7, 8])
    X = X[labeled_idx, :]
    Z, _ = preprocess(test_data[1:, :], onehot_cols=[1, 5, 7, 8])
    assert X.shape[1] == Z.shape[1]
    features = list(data[0, 1:]) + onehot_features

else:
    raise NotImplementedError("Dataset %s not handled" % dataset)

print("Features", features)
print("Train/test size", X.shape, Z.shape)

Preprocessing the titanic dataset

Features [np.str_('pclass'), np.str_('sex'), np.str_('age'), np.str_('sibsp'), np.str_('parch'), np.str_('ticket'), np.str_('fare'), np.str_('cabin'), np.str_('embarked'), np.str_('male'), np.str_('female'), np.str_('S'), np.str_('C'), np.str_('Q')]
Train/test size (1009, 14) (300, 14)


## My Decision Tree

In [9]:
# Train the best model on the Decision Tree with tuning the "max_depth" hyperparameter

depths_dt = np.arange(1, 41) # Trying max_depth from 1 to 40
best_val_acc = 0
best_depth = 0
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
for depth in depths_dt:
    params_dt = {
        "max_depth": depth,
    }
    clf_dt = DecisionTree(**params_dt)
    clf_dt.fit(X_train, y_train)
    y_val_pred = clf_dt.predict(X_val)
    val_accuracy = np.mean(y_val_pred == y_val)
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        best_depth = depth
    
print("Best decision tree val acc for titanic dataset: %.4f at depth %d" % (best_val_acc, best_depth))

Best decision tree val acc for titanic dataset: 0.8267 at depth 8


In [10]:
# my best decision tree with max_depth = 8

best_depth = 8
params_best_dt = {
    "max_depth": best_depth,
}
clf_dt = DecisionTree(**params_best_dt)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
clf_dt.fit(X_train, y_train)
y_train_pred = clf_dt.predict(X_train)
train_accuracy = np.mean(y_train_pred == y_train)
train_accs.append(train_accuracy)
y_val_pred = clf_dt.predict(X_val)
val_accuracy = np.mean(y_val_pred == y_val)
val_accs.append(val_accuracy)
print("Best decision tree train acc for titanic dataset: %.4f" % train_accuracy)
print("Best decision tree val acc for titanic dataset: %.4f" % val_accuracy)

Best decision tree train acc for titanic dataset: 0.8525
Best decision tree val acc for titanic dataset: 0.8267


## My Random Forest

In [11]:
# Train the best model on the Random Forest with hyperparameter tuning

clf_rf = RandomForest()
depths = [5, 7, 9, 11, 13, 15, 17, 20, 25, 30, 35, 40]
param_grid = {
    'params': [{'max_depth': depth} for depth in depths],
    'n': [30, 50, 100, 150, 200],
    'm': [1, 2, 3, 4, 5, 6, 8, 10]
}

grid_search = GridSearchCV(estimator=clf_rf, param_grid=param_grid, scoring='accuracy', cv=5, n_jobs=-1, verbose=1)
grid_search.fit(X, y)

print("Best params:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

Fitting 5 folds for each of 480 candidates, totalling 2400 fits
Best params: {'m': 4, 'n': 200, 'params': {'max_depth': 20}}
Best score: 0.7918575439633515


In [ ]:
# my best decision tree with max_depth = 20, n (num_trees) = 200, m (max_features) = 4
best_m, best_n, params_best_rf = grid_search.best_params_['m'], grid_search.best_params_['n'], grid_search.best_params_['params']

clf_rf = RandomForest(params=params_best_rf, n=best_n, m=best_m)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
clf_rf.fit(X_train, y_train)
y_train_pred = clf_rf.predict(X_train)
train_accuracy = np.mean(y_train_pred == y_train)
y_val_pred = clf_rf.predict(X_val)
val_accuracy = np.mean(y_val_pred == y_val)
print("Best random forest train acc for titanic dataset: %.4f" % train_accuracy)
print("Best random forest val acc for titanic dataset: %.4f" % val_accuracy)

Best random forest train acc for titanic dataset: 0.9789
Best random forest val acc for titanic dataset: 0.7921


## Kaggle

In [13]:
# Here I choose Random Forest as my classifier
clf_titanic = RandomForest(params=params_best_rf, n=best_n, m=best_m)
clf_titanic.fit(X, y)
y_test_pred = clf_titanic.predict(Z)

# save the results into a kaggle accepted csv

def results_to_csv(y_test):
    y_test = y_test.astype(int)
    df = pd.DataFrame({'Category': y_test})
    df.index += 1 # Ensures that the index starts at 1
    df.to_csv('submission_titanic.csv', index_label='Id')

results_to_csv(y_test_pred)

# Conclusion 

## Spam dataset

Decision tree train accuracy: $0.8728$

Decision tree validation accuracy: $0.8188$

Random forest train accuracy: $0.8932$

Random forest validation accuracy: $0.8250$

## Titanic dataset

Decision tree train accuracy: $0.8525$

Decision tree validation accuracy: $0.8267$

Random forest train accuracy: $0.9789$

Random forest validation accuracy: $0.7921$

## Kaggle

spam private score: $0.840$

spam public score: $0.841$

titanic private score: $0.780$

titanic public score: $0.766$